In [17]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [18]:
def process_all_pdfs(pdf_directory):
    all_documents=[]
    pdf_dir=Path(pdf_directory)

    pdf_files=list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()

            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(all_documents)} pages")

        except Exception as e:
            print(f"Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

data_dir = Path("../data")
if not data_dir.exists():
    data_dir = Path("data")

all_pdf_documents= process_all_pdfs(str(data_dir))

Found 4 PDF files to process

Processing: attention.pdf
Loaded 1 pages

Processing: embeddings.pdf
Loaded 2 pages

Processing: objectdetection.pdf
Loaded 3 pages

Processing: proposal.pdf
Loaded 4 pages

Total documents loaded: 4


In [19]:
all_pdf_documents

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content='Attention Is All You Need - Summary\nThe attention mechanism allows a model to weigh the importance of different parts of the input when producing an output. Self-attention relates different positions of a single sequence to compute a representation of that sequence.\nKey points:\n- Scaled dot-product attention computes queries, keys and values.\n- Multi-head attention runs several attention functions in parallel.\n- The Transformer architecture relies entirely on attention, removing recurrence.\n- Attention improves parallelization and reduces training cost.'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\embeddings.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'embed

In [20]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [21]:
chunks=split_documents(all_pdf_documents)
chunks

Split 4 documents into 4 chunks

Example chunk:
Content: Attention Is All You Need - Summary
The attention mechanism allows a model to weigh the importance of different parts of the input when producing an output. Self-attention relates different positions ...
Metadata: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content='Attention Is All You Need - Summary\nThe attention mechanism allows a model to weigh the importance of different parts of the input when producing an output. Self-attention relates different positions of a single sequence to compute a representation of that sequence.\nKey points:\n- Scaled dot-product attention computes queries, keys and values.\n- Multi-head attention runs several attention functions in parallel.\n- The Transformer architecture relies entirely on attention, removing recurrence.\n- Attention improves parallelization and reduces training cost.'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\embeddings.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'embed

In [22]:
import numpy as py
from sentence_transformers import SentenceTransformer
import chromadb 
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


c:\Users\HP\OneDrive\Desktop\AgenticLanggraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
class EmbeddingManager:
    def __init__(self, model_name: str="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension:{self.model.get_embedding_dimension()}")

        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 833.36it/s]


Model loaded successfully. Embedding dimension:384


In [ ]:
class VectorStore:
    def __init__(self, collection_name: str="pdf_documents", persist_directory:str="../data/vector_store"):
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
                )
            print(f"Vector store ready: {self.collection_name} at {self.persist_directory}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
        
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents)!=len(embeddings):
            raise ValueError("NUmber of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        ids= []
        metadatas= []
        documents_text= []
        embeddings_list= []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            docs_id=f"doc_{uuid.uuid4().hex[:8]}..{i}"
            ids.append(metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.toList())

            try:
                self.collection.add(
                    ids=ids,
                    embeddings=embeddings_list,
                    metadatas=metadatas,
                    documents=documents_text

                )
                print(f"Successfully added {len(documents)} documents to vector store ")
                print(f"Total documents in collection : {self.collection.count()}")

            except Exception as e:
                print(f"Error addline documents to the vector store: {e}")
